# 02 - Camada Silver
Limpeza, tipagem, nomes em português e deduplicação. **A Bronze nunca é alterada**: aqui só lemos dela e gravamos em silver.
OBS.: É necessário executar a cotação do dólar após a primeira célula, pois o financeiro depende da taxa

In [0]:
from pyspark.sql import functions as F, Window

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

def salvar(df, tabela):
    # a Camada Bronze usa append, então pode conter versões repetidas, tratamos isso aqui
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(tabela))
    print(f"OK: {tabela} -> {spark.table(tabela).count()} linhas")

def ultima_versao(df, chave):
    # mantém só o registro mais recente por chave
    w = Window.partitionBy(*chave).orderBy(F.col("ingestion_datetime").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

NUM = r"^-?\d{1,15}(\.\d+)?$"

def num_seguro(col, tipo):
    # só converte se o texto for numérico e couber no tipo de destino
    # sem interromper o pipeline
    c = F.trim(col.cast("string"))
    v = F.when(c.rlike(NUM), c.cast("decimal(38,10)"))
    if tipo == "int":
        return F.when(v.between(-2147483648, 2147483647), v.cast("int"))
    return v.cast(tipo)

silver.tb_cotacao_dolar 

In [0]:
cot = ultima_versao(spark.table("bronze.tb_cotacao_dolar"), ["dataHoraCotacao"])
cot = (cot
       .withColumn("data_cotacao", F.to_date(F.substring("dataHoraCotacao", 1, 10)))
       .withColumn("cotacao_compra", num_seguro(F.col("cotacaoCompra"), "double"))
       .filter(F.col("data_cotacao").isNotNull() & F.col("cotacao_compra").isNotNull()))

# um valor por dia: fica a última cotação do dia
w_dia = Window.partitionBy("data_cotacao").orderBy(F.col("dataHoraCotacao").desc())
cot = (cot.withColumn("_rn", F.row_number().over(w_dia)).filter("_rn = 1")
          .select("data_cotacao", "cotacao_compra"))

# calendário contínuo
lim = cot.agg(F.min("data_cotacao").alias("ini"), F.max("data_cotacao").alias("fim"))
calendario = lim.select(
    F.explode(F.sequence("ini", "fim", F.expr("interval 1 day"))).alias("data_cotacao"))

# fins de semana/feriados recebem o último valor disponível
w_ff = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
serie = (calendario.join(cot, "data_cotacao", "left")
         .withColumn("cotacao_preenchida", F.col("cotacao_compra").isNull())
         .withColumn("cotacao_compra",
                     F.last("cotacao_compra", ignorenulls=True).over(w_ff)))

salvar(serie, "silver.tb_cotacao_dolar")
display(spark.table("silver.tb_cotacao_dolar").orderBy("data_cotacao"))

# taxa usada na conversão dolar para real: a cotação mais recente da série
taxa = (spark.table("silver.tb_cotacao_dolar")
        .orderBy(F.col("data_cotacao").desc()).first()["cotacao_compra"])
print("Taxa USD->BRL:", taxa)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK: silver.tb_cotacao_dolar -> 8 linhas


data_cotacao,cotacao_compra,cotacao_preenchida
2026-09-30,5.1803,false
2026-10-01,5.2073,false
2026-10-02,5.2232,false
2026-10-03,5.2232,true
2026-10-04,5.2232,true
2026-10-05,4.9853,false
2026-10-06,4.9692,false
2026-10-07,4.9929,false


Taxa USD->BRL: 4.9929


silver.tb_info_filmes

In [0]:
b = ultima_versao(spark.table("bronze.tb_movies_info").filter(F.col("id").isNotNull()), ["id"])

# status: normaliza e só depois traduz
s = F.trim(F.regexp_replace(F.lower(F.trim(F.col("status"))), r"[^a-z]+", " "))
status_map = {
    "released": "Lançado", "post production": "Pós-Produção", "postproduction": "Pós-Produção",
    "in production": "Em Produção", "planned": "Planejado", "rumored": "Rumores",
    "rumoured": "Rumores", "canceled": "Cancelado", "cancelled": "Cancelado",
}
status_pt = F.lit("Não Informado")          # corrompido / não mapeável
for k, v in status_map.items():
    status_pt = F.when(s == k, F.lit(v)).otherwise(status_pt)

# A ordem importa: o coalesce usa o 1º formato que converter. Em datas ambíguas (ex.: 03/04/2020)
# prevalece dia/mês (d/M/yyyy). Só vira NULL se NENHUM formato servir.
FORMATOS = ["yyyy-M-d", "yyyy/M/d", "d/M/yyyy", "M/d/yyyy", "d-M-yyyy", "M-d-yyyy",
            "yyyy-MM-dd HH:mm:ss", "d MMM yyyy", "MMM d, yyyy", "MMMM d, yyyy",
            "d MMMM yyyy", "yyyyMMdd"]
data = F.coalesce(*[F.expr(f"to_date(try_to_timestamp(trim(release_date), '{f}'))")
                    for f in FORMATOS])

info = (b.select(
            F.col("id").alias("id_filme"),
            F.col("title").alias("titulo"),
            F.col("original_title").alias("titulo_original"),
            data.alias("data_lancamento"),
            num_seguro(F.col("runtime"), "int").alias("duracao_minutos"),
            F.col("original_language").alias("idioma_original"),
            status_pt.alias("status_filme"),
            F.col("overview").alias("sinopse"),
            F.col("tagline").alias("frase_divulgacao"))
        .withColumn("ano_lancamento", F.year("data_lancamento")))

salvar(info, "silver.tb_info_filmes")
display(info.groupBy("status_filme").count())

OK: silver.tb_info_filmes -> 97611 linhas


status_filme,count
Planejado,47
Lançado,96261
Pós-Produção,697
Em Produção,604
Não Informado,2


silver.tb_financeiro_filmes

In [0]:
def limpa_moeda(col):
    c = F.lower(F.trim(col.cast("string")))
    # textos que representam ausência de dado viram NULL ANTES da conversão de tipo
    c = F.when(c.isin("unknown", "não informado", "nao informado", "n/a", "null", "none", ""),
               F.lit(None)).otherwise(c)
    # remove $, US$, R$, espaços e textos
    c = F.regexp_replace(c, r"[^0-9.,\-]", "")
    # "1.000.000" -> "1000000"
    c = F.when(c.rlike(r"^\d{1,3}(\.\d{3})+$"), F.regexp_replace(c, r"\.", "")).otherwise(c)
    # "1,000,000" -> "1000000"
    c = F.regexp_replace(c, r",(?=\d{3}(\D|$))", "")
    c = F.regexp_replace(c, ",", ".")
    v = num_seguro(c, "decimal(18,2)")
    return F.when(v > 0, v)       

f = ultima_versao(spark.table("bronze.tb_movies_financials").filter(F.col("id").isNotNull()), ["id"])

fin = (f.select(F.col("id").alias("id_filme"),
                limpa_moeda(F.col("budget")).alias("orcamento_usd"),
                limpa_moeda(F.col("revenue")).alias("receita_usd"))
# Conversão para BRL: usa a cotação mais recente da série (variável `taxa`, definida na célula de silver.tb_cotacao_dolar, que por isso precisa rodar ANTES desta). Uma única taxa para toda a base mantém todos os filmes comparáveis na mesma data de referência.
       .withColumn("orcamento_brl", (F.col("orcamento_usd").cast("double") * F.lit(taxa)).cast("decimal(18,2)"))
       .withColumn("receita_brl",   (F.col("receita_usd").cast("double") * F.lit(taxa)).cast("decimal(18,2)"))
       # lucro: se faltar orçamento OU receita, o lucro fica NULL
       .withColumn("lucro_usd", F.col("receita_usd") - F.col("orcamento_usd"))
       .withColumn("lucro_brl", F.col("receita_brl") - F.col("orcamento_brl"))
       # margem % só quando o orçamento é > 0 -> sem divisão por zero
       .withColumn("margem_lucro_pct",
                   F.when(F.col("orcamento_usd") > 0,
                          F.round(F.col("lucro_usd") / F.col("orcamento_usd") * 100, 2))))

salvar(fin, "silver.tb_financeiro_filmes")
display(fin.limit(10))

OK: silver.tb_financeiro_filmes -> 99006 linhas


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
1000004,null,null,null,null,null,null,null
1000005,null,null,null,null,null,null,null
1000011,null,null,null,null,null,null,null
1000030,null,null,null,null,null,null,null
1000073,6000000.00,null,29957400.00,null,null,null,null
1000081,5.00,null,24.96,null,null,null,null
1000091,null,null,null,null,null,null,null
1000092,null,null,null,null,null,null,null
1000094,null,null,null,null,null,null,null
1000095,null,null,null,null,null,null,null


silver.tb_metricas_engajamento

In [0]:
m = ultima_versao(spark.table("bronze.tb_movies_metrics").filter(F.col("id").isNotNull()), ["id"])

# troca vírgula por ponto e, se sobrar mais de um ponto, mantém só o último como separador decimal.
p = F.regexp_replace(F.trim(F.col("popularity").cast("string")), r"[^0-9.,\-]", "")
p = F.regexp_replace(p, ",", ".")
p = F.regexp_replace(p, r"\.(?=.*\.)", "")
popularidade = num_seguro(p, "double")

# Notas fora de 0-10 (inclusive erro de escala) e contagens negativas viram NULL (regra de negócio)

def nota(col):
    v = num_seguro(col, "double")
    return F.when((v >= 0) & (v <= 10), v)      

def contagem(col):
    v = num_seguro(col, "int")
    return F.when(v >= 0, v)                    

met = m.select(
    F.col("id").alias("id_filme"),
        # popularidade acima de 10.000 é erro de escala/formatação na origem -> NULL
    F.when((popularidade >= 0) & (popularidade <= 10000), popularidade).alias("popularidade"),
    nota(F.col("vote_average")).alias("nota_media_tmdb"),
    contagem(F.col("vote_count")).alias("qtd_votos_tmdb"),
    nota(F.col("averageRating")).alias("nota_media_imdb"),
    contagem(F.col("numVotes")).alias("qtd_votos_imdb"))

salvar(met, "silver.tb_metricas_engajamento")
display(met.describe())

OK: silver.tb_metricas_engajamento -> 95115 linhas


summary,id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
count,95115,91653,91443,87449,82897,85015
mean,736592.0760763286,3.8152335962818893,3.752087453386259,55.76330203890267,6.239002014548176,3072.492171969652
stddev,256369.1710004834,44.30384213851323,3.29620251231039,530.363757529106,1.4312164637604825,29260.85645486253
min,1000058,0.0,0.0,0,0.0,0
max,999999,5003.0,10.0,28894,10.0,1735566


 silver.tb_avaliacoes_usuarios

In [0]:
r = (spark.table("bronze.tb_movies_reviews")
     .dropDuplicates(["id", "nome", "nota", "comentario"]))   

nota_u = num_seguro(F.col("nota"), "double")
aval = r.select(
    F.col("id").alias("id_filme"),
    F.col("nome").alias("nome_usuario"),
    F.when((nota_u >= 0) & (nota_u <= 10), nota_u).alias("nota_usuario"),
    F.when(F.trim(F.col("comentario")).isNull() | (F.trim(F.col("comentario")) == ""),
           F.lit("Sem comentário")).otherwise(F.col("comentario")).alias("comentario_usuario"))

salvar(aval, "silver.tb_avaliacoes_usuarios")

OK: silver.tb_avaliacoes_usuarios -> 32412 linhas


silver.tb_generos e silver.tb_pessoas_empresas

In [0]:
cred = ultima_versao(spark.table("bronze.tb_credits_and_tags").filter(F.col("id").isNotNull()), ["id"])

def explode_lista(df, coluna, separadores=r"[;,]", limpa_colchetes=False):
    # unifica os separadores (vírgula, ponto e vírgula e, p/ gêneros, também "|") no próprio split
    texto = F.col(coluna).cast("string")
    if limpa_colchetes:
        texto = F.regexp_replace(texto, r"""[\[\]'"]""", "")
    lista = F.split(texto, separadores)
    return (df.select(F.col("id").alias("id_filme"), F.explode(lista).alias("valor"))
              .withColumn("valor", F.trim(F.col("valor"))))

def valido(c, max_len):
    # remove vazios, números deslocados e textos longos
    return (c.isNotNull() & (F.length(c) > 1) & (F.length(c) <= max_len)
            & ~c.rlike(r"^[\d\s.\-]+$"))

# Domínio de negócio: só os gêneros oficiais do TMDB.
# Qualquer outro texto (países, idiomas, frases, nomes) veio do Column Shift da origem e é descartado.
GENEROS_VALIDOS = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
                   "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
                   "Romance", "Science Fiction", "Tv Movie", "Thriller", "War", "Western"]

generos = (explode_lista(cred, "genres", separadores=r"[;,|]", limpa_colchetes=True)
           .select("id_filme", F.initcap("valor").alias("genero"))
           .filter(F.col("genero").isin(GENEROS_VALIDOS))
           .dropDuplicates())
salvar(generos, "silver.tb_generos")
display(generos.groupBy("genero").count().orderBy(F.desc("count")))

tipos = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista",
         "production_companies": "Produtora"}
partes = [explode_lista(cred, col).filter(valido(F.col("valor"), 60))
          .select("id_filme", F.initcap("valor").alias("nome_entidade"),
                  F.lit(tipo).alias("tipo_entidade"))
          for col, tipo in tipos.items()]
pessoas = partes[0]
for p_ in partes[1:]:
    pessoas = pessoas.unionByName(p_)
pessoas = pessoas.dropDuplicates()

salvar(pessoas, "silver.tb_pessoas_empresas")
display(pessoas.groupBy("tipo_entidade").count())

OK: silver.tb_generos -> 141937 linhas


genero,count
Drama,32615
Documentary,19234
Comedy,18825
Thriller,10400
Horror,9833
Romance,7716
Action,6114
Crime,4786
Animation,4524
Tv Movie,4115


OK: silver.tb_pessoas_empresas -> 905666 linhas


tipo_entidade,count
Produtora,120416
Ator,541273
Diretor,107389
Roteirista,136588
